# PlantSeg -- Array 2nd-Revision Notebook A: Single-Seed Diagnostics (seed 42)

**Pure inference. No training, no new checkpoints.** Every analysis below reuses the
original SEED=42 checkpoints that already back Table 8/9/10 and Section 4/5/6 of the paper.

## What this notebook answers

1. **Section 1 -- Chance-baseline faithfulness check** (R3/R4/R5): the paper calls the
   classifier's and segmenters' Grad-CAM/attention-rollout "faithful" using a CAM-GT IoU
   that is thresholded at the *ground-truth* lesion-area fraction. Reviewers want to know
   this is not an artifact of that threshold choice. This section adds a **chance baseline**
   (a random attribution map, same threshold rule) and a paired Wilcoxon test, real vs.
   chance, for the classifier and all 6 segmenters.
2. **Section 2 -- TTA vs. single-pass breakdown, all 8 classifiers** (R4 point 3 / R5 point
   4): round 1 already disclosed that the headline classifier numbers use horizontal-flip
   test-time augmentation (TTA) while the deployed pipeline uses a single forward pass. This
   section adds the granular breakdown reviewers now want: per-view accuracy (original vs.
   flipped), the two views' agreement rate, and counts of images TTA *corrects* vs. *harms*,
   for every one of the 8 classifiers (not just the deployed one).
3. **Section 3 -- Leakage dedup + recompute on the full 311-image union** (R5 point 2): the
   paper's clean-vs-leaked comparison only used the 250-image URL-leak list. This section
   reconstructs the full 311-image union (250 URL-leak + 213 near-duplicate-audit, 152
   overlap) and extends the comparison from classification accuracy alone to classification,
   segmentation fg-IoU, severity MAE/CCC, and a confidence-flag re-validation restricted to
   the clean subset only.
4. **Section 4 -- Minor lookups, no computation needed**: documents, with pointers to the
   exact code cells responsible, why R5 point 1(d) (how the CAM threshold is chosen) and R5
   minor comment (c) (the deletion-AUC pixel-aggregation rule) are already answered by the
   existing deployed-pipeline code and need no new experiment.

## Required Kaggle inputs (attach as datasets)
- The PlantSeg image + annotation dataset (`Metadata.csv`, `images/`,
  `annotation_{train,val,test}.json`).
- The 8 original SEED=42 classifier checkpoints (`*_best.pt`) -- the same dataset already
  used by Stage 6 (`CLS_CKPT_DIR` below).
- The 6 original SEED=42 segmenter checkpoints (`*.pt`) -- the same dataset already used by
  Stage 4/6 (`SEG_CKPT_DIR` below).

No new dataset needs to be created for this notebook -- it only reads checkpoints that are
already attached to existing notebooks.

## Estimated runtime
Chance-baseline section is the heaviest piece (300-image subset x 7 models x 3 random
repeats, with a 10-step deletion sweep per repeat) but is still pure inference -- expect
roughly **1.5-2.5 hours total** for all 4 sections combined on a Kaggle T4/P100, comfortably
inside the 12-hour session limit. Each section writes its own CSV/JSON before moving on, so
if a session is interrupted partway through, only the remaining sections need to be re-run.

## Setup

In [1]:
!pip install -q timm segmentation-models-pytorch grad-cam transformers scikit-image
import os, json, glob, random, warnings, io, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import timm
import segmentation_models_pytorch as smp
from transformers import SegformerForSemanticSegmentation
from pytorch_grad_cam import GradCAM
from scipy.stats import wilcoxon, fisher_exact, mannwhitneyu
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00
Device: cuda


## Config -- edit the `CKPT_DIR` paths below only if your dataset mount points differ

In [2]:
# ============ CONFIG ============
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

def find_ckpt_dir(required_names):
    for d in glob.glob("/kaggle/input/*"):
        if all((Path(d)/n).exists() for n in required_names): return Path(d)
    # fall back to locating any one of the files and using its parent
    for n in required_names:
        hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        if hits: return Path(hits[0]).parent
    raise FileNotFoundError(f"Could not find a dataset containing {required_names}")

CLS_CKPT_DIR = find_ckpt_dir(["ConvNeXtV2Tiny_best.pt", "MobileNetV3Large_best.pt"])
SEG_CKPT_DIR = find_ckpt_dir(["DeepLabV3Plus_efficientnet-b3.pt", "UNet_resnet34.pt"])
print("CLS_CKPT_DIR:", CLS_CKPT_DIR)
print("SEG_CKPT_DIR:", SEG_CKPT_DIR)

CLS_CKPTS = {
    "MobileNetV3Large":  CLS_CKPT_DIR/"MobileNetV3Large_best.pt",
    "EfficientNetB0":    CLS_CKPT_DIR/"EfficientNetB0_best.pt",
    "ResNet50":          CLS_CKPT_DIR/"ResNet50_best.pt",
    "EfficientNetV2S":   CLS_CKPT_DIR/"EfficientNetV2S_best.pt",
    "ConvNeXtV2Tiny":    CLS_CKPT_DIR/"ConvNeXtV2Tiny_best.pt",
    "ViTBase16":         CLS_CKPT_DIR/"ViTBase16_best.pt",
    "SwinTiny":          CLS_CKPT_DIR/"SwinTiny_best.pt",
    "GatedFusionLight":  CLS_CKPT_DIR/"GatedFusionLight_best.pt",
}
SEG_CKPTS = {
    "UNet_resnet34":                 SEG_CKPT_DIR/"UNet_resnet34.pt",
    "UNetPP_resnet50":               SEG_CKPT_DIR/"UNetPP_resnet50.pt",
    "FPN_resnet50":                  SEG_CKPT_DIR/"FPN_resnet50.pt",
    "DeepLabV3Plus_resnet50":        SEG_CKPT_DIR/"DeepLabV3Plus_resnet50.pt",
    "DeepLabV3Plus_efficientnet-b3": SEG_CKPT_DIR/"DeepLabV3Plus_efficientnet-b3.pt",
    "SegFormer_b2":                  SEG_CKPT_DIR/"SegFormer_b2.pt",
}
CLS_CKPT = CLS_CKPTS["ConvNeXtV2Tiny"]            # the deployed classifier
SEG_CKPT_BEST = SEG_CKPTS["DeepLabV3Plus_efficientnet-b3"]  # the deployed segmenter

OUTDIR = Path("/kaggle/working/r2_notebookA_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS = 384      # CNN classifiers (384x384, per the corrected Methods text)
IMG_SIZE_CLS_TF = 224   # ViT-B/16 / Swin-Tiny / GatedFusionLight are fixed at 224x224
IMG_SIZE_SEG = 512      # segmenters
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["maskpath"] = meta.apply(lambda r: DATA_ROOT/"annotations"/r["split"]/r["Label file"], axis=1) \
    if "Label file" in meta.columns else None
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)
print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

# If "Label file"/maskpath did not resolve (column name differs), rasterize masks from the
# COCO-style annotation_*.json files instead -- same approach as Stage 3/4/6.
if meta["maskpath"].isna().any() or not meta["maskpath"].apply(lambda p: p is not None and Path(p).exists()).all():
    from collections import defaultdict
    from PIL import ImageDraw
    MASK = Path("/kaggle/working/masks")
    def load_anns(split):
        d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
        for a in d["annotations"]: by[a["image_id"]].append(a)
        fn = {}
        for im in d["images"]:
            polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
            fn[im["file_name"]] = (im["width"], im["height"], polys)
        return fn
    def rasterize(w, h, polys):
        m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
        for seg in polys: dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
        return m
    mp_by_name = {}
    for split in ["train", "val", "test"]:
        fn = load_anns(split); od = MASK/split; od.mkdir(parents=True, exist_ok=True)
        sub = meta[meta.split == split]
        for r in sub.itertuples():
            w, h, polys = fn.get(r.Name, (None, None, []))
            mp = od/(Path(r.Name).stem + ".png")
            if not mp.exists():
                if w is None:
                    with Image.open(r.filepath) as i0: w, h = i0.size
                rasterize(w, h, polys).save(mp)
            mp_by_name[r.Name] = mp
    meta["maskpath"] = meta["Name"].map(mp_by_name)
    train_df = meta[meta.split=="train"].reset_index(drop=True)
    val_df   = meta[meta.split=="val"].reset_index(drop=True)
    test_df  = meta[meta.split=="test"].reset_index(drop=True)
    print("Rasterized masks from annotation_*.json (maskpath column populated).")

def load_state(model, ckpt_path):
    raw = torch.load(ckpt_path, map_location="cpu")
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    raw = {k.replace("module.", ""): v for k, v in raw.items()}
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        print(f"  [load_state:{Path(ckpt_path).name}] missing={len(missing)} unexpected={len(unexpected)}")
        total_model_keys = len(model.state_dict())
        if total_model_keys and len(missing) >= total_model_keys * 0.5:
            print(f"  [load_state:{Path(ckpt_path).name}] sample checkpoint keys: {list(raw.keys())[:3]}")
            print(f"  [load_state:{Path(ckpt_path).name}] sample expected keys  : {list(model.state_dict().keys())[:3]}")
            raise RuntimeError(f"{Path(ckpt_path).name}: >=50% of keys unmatched "
                                f"(missing={len(missing)}, unexpected={len(unexpected)}) -- "
                                f"checkpoint did not load; refusing to report numbers from random weights")
    return model

def build_classifier(num_classes):
    return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes)

SMP_SPECS = {
    "UNet_resnet34":                 (smp.Unet,          "resnet34"),
    "UNetPP_resnet50":               (smp.UnetPlusPlus,  "resnet50"),
    "DeepLabV3Plus_resnet50":        (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50":                  (smp.FPN,           "resnet50"),
}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}

class SegNet(nn.Module):
    def __init__(self, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            self.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); self.kind = "smp"
        else:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
    def forward(self, x):
        if self.kind == "smp": return self.net(x)
        lg = self.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

cls_tf = T.Compose([T.Resize((IMG_SIZE_CLS, IMG_SIZE_CLS)), T.ToTensor(), T.Normalize(MEAN, STD)])
seg_eval_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)

print("Config ready.")

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
CLS_CKPT_DIR: /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints
SEG_CKPT_DIR: /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints
classes=115  train=5367  val=846  test=1561
Config ready.


## Section 1 -- Chance-baseline faithfulness check

Same 300-image stratified subset used throughout the paper's XAI analyses (Stage 4, Stage 6
Section 1/4), so every number below is directly comparable to Table 10. For each image we
compute the **real** Grad-CAM/attention-rollout faithfulness metrics (identical functions to
Stage 4/6) and a **chance** baseline: a uniform-random attribution map run through the exact
same thresholding/deletion-AUC code. If "real" beats "chance" with a small paired-test
p-value, the faithfulness numbers are not an artifact of the GT-area-matched threshold rule.

In [3]:
# ============ SECTION 1: CHANCE-BASELINE FAITHFULNESS CHECK ============
NAMES_300 = ['apple_black_rot_google_0037.jpg', 'apple_black_rot_google_0077.jpg', 'apple_mosaic_virus_57.jpg', 'apple_mosaic_virus_google_0074.jpg', 'apple_rust_147.jpg', 'apple_rust_google_0007.jpg', 'apple_rust_google_0011.jpg', 'apple_scab_50.jpg', 'apple_scab_85.jpg', 'apple_scab_google_0018.jpg', 'apple_scab_google_0031.jpg', 'apple_scab_google_0153.jpg', 'apple_scab_google_0244.jpg', 'banana_anthracnose_Baidu_0059.jpg', 'banana_anthracnose_Bing_0006.jpg', 'banana_anthracnose_Bing_0022.jpg', 'banana_anthracnose_Bing_0032.jpg', 'banana_black_leaf_streak_Bing_0124.jpg', 'banana_black_leaf_streak_banana black sigatoka (26).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_bunchy_top_Google_0037.jpg', 'banana_cigar_end_rot_Bing_0007.jpg', 'banana_cigar_end_rot_Google_0101.jpg', 'banana_cordana_leaf_spot_Bing_0007.jpg', 'banana_cordana_leaf_spot_Bing_0037.jpg', 'banana_panama_disease_126.jpg', 'banana_panama_disease_google_0064.jpg', 'basil_downy_mildew_google_0031.jpg', 'basil_downy_mildew_google_0091.jpg', 'bean_halo_blight_11.jpg', 'bean_halo_blight_2.jpg', 'bean_mosaic_virus_24.jpg', 'bean_mosaic_virus_google_0098.jpg', 'bean_rust_google_0045.jpg', 'bean_rust_google_0068.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0028.jpg', 'bell_pepper_blossom_end_rot_Bing_0002.jpg', 'bell_pepper_blossom_end_rot_Bing_0004.jpg', 'bell_pepper_blossom_end_rot_Bing_0077.jpg', 'bell_pepper_frogeye_leaf_spot_Bing_0008.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0049.jpg', 'bell_pepper_powdery_mildew_Bing_0035.jpg', 'bell_pepper_powdery_mildew_Google_0118.jpg', 'blueberry_anthracnose_Bing_0006.jpg', 'blueberry_anthracnose_Bing_0145.jpg', 'blueberry_botrytis_blight_Bing_0015.jpg', 'blueberry_botrytis_blight_Bing_0021.jpg', 'blueberry_mummy_berry_Bing_0024.jpg', 'blueberry_mummy_berry_Google_0038.jpg', 'blueberry_rust_google_0004.jpg', 'blueberry_rust_google_0053.jpg', 'blueberry_scorch_Bing_0011.jpg', 'blueberry_scorch_Bing_0031.jpg', 'blueberry_scorch_Bing_0062.jpg', 'broccoli_alternaria_leaf_spot_Bing_0000.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_downy_mildew_30.jpg', 'broccoli_downy_mildew_40.jpg', 'broccoli_ring_spot_Bing_0002.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_3.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0002.jpg', 'cabbage_black_rot_Bing_0018.jpg', 'cabbage_black_rot_Bing_0039.jpg', 'cabbage_black_rot_Bing_0080.jpg', 'cabbage_black_rot_Bing_0114.jpg', 'cabbage_downy_mildew_Bing_0012.jpg', 'cabbage_downy_mildew_Bing_0124.jpg', 'carrot_alternaria_leaf_blight_Bing_0046.jpg', 'carrot_alternaria_leaf_blight_Bing_0071.jpg', 'carrot_cavity_spot_google_0000.jpg', 'carrot_cavity_spot_google_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'carrot_cercospora_leaf_blight_Bing_0212.jpg', 'cauliflower_alternaria_leaf_spot_google_0003.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'cauliflower_bacterial_soft_rot_Bing_0205.jpg', 'celery_anthracnose_google_0007.jpg', 'celery_anthracnose_google_0035.jpg', 'celery_early_blight_52.jpg', 'celery_early_blight_google_0001.jpg', 'celery_early_blight_google_0009.jpg', 'cherry_leaf_spot_google_0180.jpg', 'cherry_leaf_spot_google_0216.jpg', 'cherry_powdery_mildew_107.jpg', 'cherry_powdery_mildew_2.jpg', 'citrus_canker_13.jpg', 'citrus_canker_147.jpg', 'citrus_canker_163.jpg', 'citrus_canker_247.jpg', 'citrus_canker_363.jpg', 'citrus_canker_76.jpg', 'citrus_canker_89.jpg', 'citrus_canker_google_0071.jpg', 'citrus_greening_disease_29.jpg', 'citrus_greening_disease_48.jpg', 'citrus_greening_disease_google_0120.jpg', 'citrus_greening_disease_google_0251.jpg', 'coffee_berry_blotch_Bing_0027.jpg', 'coffee_berry_blotch_Bing_0114.jpg', 'coffee_berry_blotch_Bing_0120.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_15.jpg', 'coffee_leaf_rust_google_0016.jpg', 'coffee_leaf_rust_google_0135.jpg', 'corn_gray_leaf_spot_31.jpg', 'corn_gray_leaf_spot_35.jpg', 'corn_gray_leaf_spot_google_0074.jpg', 'corn_northern_leaf_blight_101.jpg', 'corn_northern_leaf_blight_14.jpg', 'corn_northern_leaf_blight_81.jpg', 'corn_rust_80.jpg', 'corn_rust_google_0078.jpg', 'corn_smut_google_0013.jpg', 'corn_smut_google_0063.jpg', 'corn_smut_google_0079.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_156.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_angular_leaf_spot_41.jpg', 'cucumber_angular_leaf_spot_google_0043.jpg', 'cucumber_bacterial_wilt_109.jpg', 'cucumber_bacterial_wilt_313.jpg', 'cucumber_bacterial_wilt_50.jpg', 'cucumber_powdery_mildew_58.jpg', 'cucumber_powdery_mildew_70.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0159.jpg', 'eggplant_cercospora_leaf_spot_12.jpg', 'eggplant_cercospora_leaf_spot_33.jpg', 'eggplant_cercospora_leaf_spot_40.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0016.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0261.jpg', 'eggplant_phytophthora_blight_Bing_0000.jpg', 'eggplant_phytophthora_blight_Bing_0002.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_leaf_blight_google_0004.jpg', 'garlic_leaf_blight_google_0210.jpg', 'garlic_rust_159.jpg', 'garlic_rust_18.jpg', 'ginger_leaf_spot_11.jpg', 'ginger_leaf_spot_google_0001.jpg', 'ginger_sheath_blight_53.jpg', 'ginger_sheath_blight_89.jpg', 'grape_black_rot_12.jpg', 'grape_black_rot_216.jpg', 'grape_black_rot_google_0013.jpg', 'grape_downy_mildew_251.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_google_0032.jpg', 'grape_downy_mildew_google_0052.jpg', 'grape_leaf_spot_101.jpg', 'grape_leaf_spot_168.jpg', 'grapevine_leafroll_disease_google_0001.jpg', 'grapevine_leafroll_disease_google_0181.jpg', 'lettuce_downy_mildew_google_0015.jpg', 'lettuce_downy_mildew_google_0027.jpg', 'lettuce_mosaic_virus_16.jpg', 'lettuce_mosaic_virus_24.jpg', 'maple_tar_spot_google_0005.jpg', 'maple_tar_spot_google_0089.jpg', 'peach_anthracnose_Bing_0002.jpg', 'peach_anthracnose_Bing_0013.jpg', 'peach_brown_rot_Bing_0000.jpg', 'peach_brown_rot_Bing_0092.jpg', 'peach_brown_rot_Bing_0114.jpg', 'peach_leaf_curl_34.jpg', 'peach_leaf_curl_google_0020.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_leaf_curl_google_0074.jpg', 'peach_rust_Bing_0001.jpg', 'peach_scab_Bing_0007.jpg', 'peach_scab_Bing_0083.jpg', 'plum_bacterial_spot_Bing_0013.jpg', 'plum_bacterial_spot_Bing_0082.jpg', 'plum_brown_rot_Bing_0001.jpg', 'plum_brown_rot_Bing_0097.jpg', 'plum_pocket_disease_google_0028.jpg', 'plum_pocket_disease_google_0032.jpg', 'plum_pocket_disease_google_0117.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Bing_0054.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0024.jpg', 'plum_rust_Google_0036.jpg', 'potato_early_blight_6.jpg', 'potato_early_blight_google_0095.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_36.jpg', 'potato_late_blight_96.jpg', 'raspberry_fire_blight_Bing_0018.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_gray_mold_Bing_0000.jpg', 'raspberry_gray_mold_Bing_0001.jpg', 'raspberry_leaf_spot_Bing_0073.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0183.jpg', 'rice_blast_151.jpg', 'rice_blast_87.jpg', 'rice_sheath_blight_148.jpg', 'rice_sheath_blight_google_0027.jpg', 'soybean_bacterial_blight_Bing_0057.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Baidu_0010.jpg', 'soybean_brown_spot_Bing_0353.jpg', 'soybean_downy_mildew_Bing_0078.jpg', 'soybean_downy_mildew_Bing_0143.jpg', 'soybean_frog_eye_leaf_spot_Bing_0141.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_frog_eye_leaf_spot_Google_0221.jpg', 'soybean_mosaic_Baidu_0059.jpg', 'soybean_mosaic_Baidu_0089.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'soybean_rust_Bing_0075.jpg', 'soybean_rust_Bing_0096.jpg', 'squash_powdery_mildew_129.jpg', 'squash_powdery_mildew_23.jpg', 'squash_powdery_mildew_6.jpg', 'squash_powdery_mildew_google_0105.jpg', 'strawberry_anthracnose_5.jpg', 'strawberry_anthracnose_google_0001.jpg', 'strawberry_leaf_scorch_google_0037.jpg', 'strawberry_leaf_scorch_google_0133.jpg', 'tobacco_blue_mold_Bing_0024.jpg', 'tobacco_blue_mold_Bing_0096.jpg', 'tobacco_brown_spot_Bing_0001.jpg', 'tobacco_brown_spot_Bing_0047.jpg', 'tobacco_brown_spot_Bing_0055.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_frogeye_leaf_spot_Google_0052.jpg', 'tobacco_mosaic_virus_117.jpg', 'tobacco_mosaic_virus_4.jpg', 'tomato_bacterial_leaf_spot_120.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0177.jpg', 'tomato_late_blight_10.jpg', 'tomato_late_blight_182.jpg', 'tomato_late_blight_205.jpg', 'tomato_leaf_mold_10.jpg', 'tomato_leaf_mold_148.jpg', 'tomato_mosaic_virus_15.jpg', 'tomato_mosaic_virus_google_0049.jpg', 'tomato_septoria_leaf_spot_10.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_80.jpg', 'tomato_septoria_leaf_spot_google_0027.jpg', 'tomato_septoria_leaf_spot_google_0070.jpg', 'tomato_septoria_leaf_spot_google_0128.jpg', 'tomato_yellow_leaf_curl_virus_15.jpg', 'tomato_yellow_leaf_curl_virus_17.jpg', 'tomato_yellow_leaf_curl_virus_40.jpg', 'tomato_yellow_leaf_curl_virus_99.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0235.jpg', 'wheat_head_scab_Baidu_0481.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0288.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0058.jpg', 'wheat_leaf_rust_Google_0328.jpg', 'wheat_loose_smut_Bing_0109.jpg', 'wheat_loose_smut_Bing_0283.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0181.jpg', 'wheat_powdery_mildew_Bing_0014.jpg', 'wheat_powdery_mildew_Google_0014.jpg', 'wheat_septoria_blotch_Google_0115.jpg', 'wheat_septoria_blotch_google_septoria (18).jpg', 'wheat_stem_rust_Baidu_0001.jpg', 'wheat_stem_rust_Bing_0217.jpg', 'wheat_stem_rust_Google_0274.jpg', 'wheat_stripe_rust_Baidu_0143.jpg', 'wheat_stripe_rust_Bing_0007.jpg', 'wheat_stripe_rust_Bing_0257.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'zucchini_bacterial_wilt_Bing_0105.jpg', 'zucchini_bacterial_wilt_Bing_0188.jpg', 'zucchini_bacterial_wilt_Bing_0260.jpg', 'zucchini_bacterial_wilt_Google_0060.jpg', 'zucchini_downy_mildew_Bing_0035.jpg', 'zucchini_downy_mildew_Bing_0259.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0155.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_yellow_mosaic_virus_11.jpg', 'zucchini_yellow_mosaic_virus_google_0038.jpg', 'zucchini_yellow_mosaic_virus_google_0067.jpg']
print(f"Reusing the exact {len(NAMES_300)}-image stratified subset from Stage 4/6")

sub_df = test_df[test_df["Name"].isin(NAMES_300)].reset_index(drop=True)
assert len(sub_df) == len(NAMES_300), f"matched {len(sub_df)} / {len(NAMES_300)} -- check Name vs image column"

N_CHANCE_REPEATS = 3
rng_master = np.random.RandomState(SEED)
DEL_STEPS = 10

def cam_gt_metrics(cam, gt_mask):
    if gt_mask.sum() == 0: return {"iou": np.nan, "pointing_hit": np.nan}
    thresh = np.quantile(cam, 1 - gt_mask.mean())
    cam_bin = cam >= thresh
    inter = (cam_bin & gt_mask).sum(); union = (cam_bin | gt_mask).sum()
    iou = float(inter/union) if union > 0 else np.nan
    py, px = np.unravel_index(np.argmax(cam), cam.shape)
    return {"iou": iou, "pointing_hit": bool(gt_mask[py, px])}

def deletion_auc_generic(prob_fn, x, cam, steps=DEL_STEPS):
    H, W = x.shape[-2:]; order = np.argsort(-cam.ravel()); npix = order.size
    baseline = x.mean(dim=(2,3), keepdim=True); fracs = np.linspace(0, 1, steps+1); probs = []
    for f in fracs:
        k = int(f*npix); xin = x.clone()
        if k > 0:
            mask = torch.zeros(npix, dtype=torch.bool, device=x.device)
            mask[order[:k]] = True; mask = mask.reshape(H, W)
            for c in range(xin.shape[1]): xin[0,c][mask] = baseline[0,c,0,0]
        probs.append(prob_fn(xin))
    return float(np.trapz(probs, fracs))

chance_rows = []

# ---- 1a. Classifier (ConvNeXtV2-Tiny), via pytorch_grad_cam -- same as Stage 5/6 ----
clf = build_classifier(NUM_CLASSES); clf = load_state(clf, CLS_CKPT).to(device).eval()
cam_extractor = GradCAM(model=clf, target_layers=[clf.stages[-1]])

for r in sub_df.itertuples():
    img = Image.open(r.filepath).convert("RGB")
    gt = np.array(Image.open(r.maskpath)); gt_bin = (gt > 0)
    x = cls_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_idx = int(clf(x).argmax(1))
    cam = cam_extractor(input_tensor=x, targets=None)[0]
    cam_rs = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(gt_bin.shape[::-1])) / 255.0
    real_m = cam_gt_metrics(cam_rs, gt_bin)

    def prob_fn(xin, _pred_idx=pred_idx):
        with torch.no_grad():
            return float(torch.softmax(clf(xin), 1)[0, _pred_idx])
    cam_for_del = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE_CLS, IMG_SIZE_CLS)))/255.0
    real_delauc = deletion_auc_generic(prob_fn, x, cam_for_del)

    rand_ious=[]; rand_hits=[]; rand_delaucs=[]
    for rep in range(N_CHANCE_REPEATS):
        rm = cam_gt_metrics(rng_master.rand(*gt_bin.shape), gt_bin)
        rand_ious.append(rm["iou"]); rand_hits.append(rm["pointing_hit"])
        rand_delaucs.append(deletion_auc_generic(prob_fn, x, rng_master.rand(*cam_for_del.shape)))

    chance_rows.append({"model": "ConvNeXtV2Tiny_classifier", "image": r.Name,
                         "real_iou": real_m["iou"], "real_pointing_hit": real_m["pointing_hit"],
                         "real_deletion_auc": real_delauc,
                         "chance_iou_mean": np.nanmean(rand_ious),
                         "chance_pointing_hit_mean": np.nanmean(rand_hits),
                         "chance_deletion_auc_mean": np.nanmean(rand_delaucs)})
del clf, cam_extractor; torch.cuda.empty_cache()
print("Classifier chance-baseline done.")

# ---- 1b. Six segmenters -- Grad-CAM (5 CNNs) / attention-rollout (SegFormer-B2), same as Stage 4/6 ----
class _GradCAMHook:
    def __init__(self, module):
        self.activations = None; self.gradients = None
        self.fh = module.register_forward_hook(self._fwd)
        self.bh = module.register_full_backward_hook(self._bwd)
    def _fwd(self, m, inp, out): self.activations = out
    def _bwd(self, m, gi, go):  self.gradients  = go[0]
    def remove(self): self.fh.remove(); self.bh.remove()

def gradcam_map(model, x, nc):
    hook = _GradCAMHook(model.net.decoder)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits[:,1] if nc==2 else logits[:,1:].sum(1)
    score.sum().backward()
    acts  = hook.activations[0].detach(); grads = hook.gradients[0].detach(); hook.remove()
    w = grads.mean(dim=(1,2))
    cam = torch.relu((w[:,None,None]*acts).sum(0)); cam = cam/(cam.max()+1e-8)
    cam = F.interpolate(cam[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def segformer_attention_map(model, x):
    with torch.no_grad():
        out = model.net.segformer(pixel_values=x, output_attentions=True) \
              if hasattr(model.net, "segformer") else model.net(pixel_values=x, output_attentions=True)
    attns = out.attentions
    if not attns:
        raise RuntimeError("segformer_attention_map: model returned no attentions (output_attentions=True "
                            "had no effect) -- likely a failed/garbled checkpoint load or a transformers "
                            "version mismatch; check the load_state warning printed for this model")
    stages=[]; cur=[]; cur_n=None
    for a in attns:
        n=a.shape[-2]
        if cur_n is None or n==cur_n: cur.append(a); cur_n=n
        else: stages.append(cur); cur=[a]; cur_n=n
    stages.append(cur)
    rollout=None
    for a in stages[-1]:
        A = a[0].mean(0); A = A/A.sum(-1,keepdim=True).clamp_min(1e-8)
        A_hat = 0.5*A + 0.5*torch.eye(A.shape[0], device=A.device)
        rollout = A_hat if rollout is None else A_hat@rollout
    imp = rollout.mean(0); side = int(round(imp.shape[0]**0.5))
    grid = imp[:side*side].reshape(side,side).float(); grid = grid/(grid.max()+1e-8)
    cam  = F.interpolate(grid[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def explain(model, x, nc, is_transformer):
    return segformer_attention_map(model,x) if is_transformer else gradcam_map(model,x,nc)

for name, ckpt in SEG_CKPTS.items():
    if not ckpt.exists():
        print(f"  [skip] {name}: checkpoint not found"); continue
    try:
        model = SegNet(name, 2); model = load_state(model, ckpt).to(device).eval()
    except Exception as e:
        print(f"  [SKIP] {name}: checkpoint failed to load -- {e}"); continue
    is_tf = (name == "SegFormer_b2")
    n_ok = 0
    for r in sub_df.itertuples():
        try:
            x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
            cam = explain(model, x, 2, is_tf)
            real_m = cam_gt_metrics(cam, gt)

            def prob_fn(xin, _model=model):
                with torch.no_grad():
                    lg = _model(xin)
                    return float(torch.softmax(lg, 1)[0, 1:].sum(0).mean())
            real_delauc = deletion_auc_generic(prob_fn, x, cam)

            rand_ious=[]; rand_hits=[]; rand_delaucs=[]
            for rep in range(N_CHANCE_REPEATS):
                rm = cam_gt_metrics(rng_master.rand(*gt.shape), gt)
                rand_ious.append(rm["iou"]); rand_hits.append(rm["pointing_hit"])
                rand_delaucs.append(deletion_auc_generic(prob_fn, x, rng_master.rand(*gt.shape)))

            chance_rows.append({"model": name, "image": r.Name,
                                 "real_iou": real_m["iou"], "real_pointing_hit": real_m["pointing_hit"],
                                 "real_deletion_auc": real_delauc,
                                 "chance_iou_mean": np.nanmean(rand_ious),
                                 "chance_pointing_hit_mean": np.nanmean(rand_hits),
                                 "chance_deletion_auc_mean": np.nanmean(rand_delaucs)})
            n_ok += 1
        except Exception as e:
            print(f"  [SKIP image] {name} / {r.Name}: {e}"); continue
    del model; torch.cuda.empty_cache()
    if n_ok == 0:
        print(f"  [SKIP] {name}: 0/{len(sub_df)} images succeeded -- excluded from chance-baseline results")
    else:
        print(f"  done: {name} ({n_ok}/{len(sub_df)} images)")

chance_df = pd.DataFrame(chance_rows)
chance_df.to_csv(OUTDIR/"chance_baseline_per_image.csv", index=False)

summary_rows=[]
for name, grp in chance_df.groupby("model"):
    row = {"model": name, "n": len(grp)}
    for real_col, chance_col, metric in [("real_iou","chance_iou_mean","iou"),
                                          ("real_pointing_hit","chance_pointing_hit_mean","pointing_hit"),
                                          ("real_deletion_auc","chance_deletion_auc_mean","deletion_auc")]:
        a = grp[real_col].dropna().values
        b = grp.loc[grp[real_col].notna(), chance_col].values
        row[f"real_mean_{metric}"] = float(np.mean(a)) if len(a) else np.nan
        row[f"chance_mean_{metric}"] = float(np.mean(b)) if len(b) else np.nan
        try:
            stat, p = wilcoxon(a, b)
            row[f"wilcoxon_p_{metric}"] = float(p)
        except Exception:
            row[f"wilcoxon_p_{metric}"] = np.nan
    summary_rows.append(row)

chance_summary = pd.DataFrame(summary_rows)
chance_summary.to_csv(OUTDIR/"chance_baseline_SUMMARY.csv", index=False)
print(chance_summary.round(4).to_string(index=False))
print()
print("Reading guide: for IoU and pointing-hit-rate, real >> chance with small p is the")
print("evidence that faithfulness is not an artifact of GT-area-matched thresholding. For")
print("deletion-AUC, real < chance (lower = more faithful) is the expected direction.")
print(f"\nSaved: {OUTDIR/'chance_baseline_per_image.csv'} and _SUMMARY.csv")

Reusing the exact 300-image stratified subset from Stage 4/6
Classifier chance-baseline done.
  done: UNet_resnet34 (300/300 images)
  done: UNetPP_resnet50 (300/300 images)
  done: FPN_resnet50 (300/300 images)
  done: DeepLabV3Plus_resnet50 (300/300 images)
  done: DeepLabV3Plus_efficientnet-b3 (300/300 images)


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 99.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.num_batches_tracked              

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

  [load_state:SegFormer_b2.pt] missing=372 unexpected=372
  [load_state:SegFormer_b2.pt] sample checkpoint keys: ['net.segformer.encoder.patch_embeddings.0.proj.weight', 'net.segformer.encoder.patch_embeddings.0.proj.bias', 'net.segformer.encoder.patch_embeddings.0.layer_norm.weight']
  [load_state:SegFormer_b2.pt] sample expected keys  : ['net.segformer.stages.0.patch_embeddings.proj.weight', 'net.segformer.stages.0.patch_embeddings.proj.bias', 'net.segformer.stages.0.patch_embeddings.layer_norm.weight']
  [SKIP] SegFormer_b2: checkpoint failed to load -- SegFormer_b2.pt: >=50% of keys unmatched (missing=372, unexpected=372) -- checkpoint did not load; refusing to report numbers from random weights
                        model   n  real_mean_iou  chance_mean_iou  wilcoxon_p_iou  real_mean_pointing_hit  chance_mean_pointing_hit  wilcoxon_p_pointing_hit  real_mean_deletion_auc  chance_mean_deletion_auc  wilcoxon_p_deletion_auc
    ConvNeXtV2Tiny_classifier 300         0.2671           

## Section 2 -- TTA vs. single-pass breakdown, all 8 classifiers

Round 1 disclosed the TTA-vs-single-pass discrepancy qualitatively for the deployed
classifier (ConvNeXtV2-Tiny: 67.6% no-TTA vs. 78.4% TTA-averaged). Round 2 asks for the
granular numbers, for every classifier: per-view accuracy, agreement between the two views,
and how many images TTA actually corrects vs. harms.

In [4]:
# ============ SECTION 2: TTA VS. SINGLE-PASS BREAKDOWN (ALL 8 CLASSIFIERS) ============
def build_backbone(name, num_classes):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights=None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes); return m, IMG_SIZE_CLS
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes), IMG_SIZE_CLS_TF
    if name == "GatedFusionLight":
        class _Gated(nn.Module):
            def __init__(self, ba, da, bb, db, nc, gh=128, hh=256, dr=0.3):
                super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
                self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
                self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
            def forward(self, x):
                fa, fb = self.branch_a(x), self.branch_b(x)
                g = self.gate(torch.cat([fa,fb], 1))
                return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))
        mb = torchvision.models.mobilenet_v3_large(weights=None)
        da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
        b = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, global_pool="avg")
        return _Gated(mb, da, b, b.num_features, num_classes), IMG_SIZE_CLS_TF
    raise ValueError(name)

CHANCE = 1.0 / NUM_CLASSES
tta_rows = []
for name, ckpt in CLS_CKPTS.items():
    if not ckpt.exists():
        print(f"  [skip] {name}: checkpoint not found"); continue
    try:
        model, img_size = build_backbone(name, NUM_CLASSES)
        model = load_state(model, ckpt).to(device).eval()
    except Exception as e:
        print(f"  [SKIP] {name}: checkpoint failed to load -- {e}"); continue
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    y_true=[]; pred_orig=[]; pred_flip=[]; pred_tta=[]
    with torch.no_grad():
        for r in test_df.itertuples():
            x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            lg_o = model(x); lg_f = model(torch.flip(x, dims=[3]))
            pred_orig.append(int(lg_o.argmax(1))); pred_flip.append(int(lg_f.argmax(1)))
            pred_tta.append(int((lg_o + lg_f).argmax(1)))
            y_true.append(label2idx[r.label])
    y_true = np.array(y_true); po = np.array(pred_orig); pf = np.array(pred_flip); pt = np.array(pred_tta)
    acc_o = (po==y_true).mean(); acc_f = (pf==y_true).mean(); acc_t = (pt==y_true).mean()
    agree = (po==pf).mean()
    corrected = int(((po!=y_true) & (pt==y_true)).sum())
    harmed    = int(((po==y_true) & (pt!=y_true)).sum())
    unchanged = int(len(y_true) - corrected - harmed)
    if acc_o < 2*CHANCE:
        print(f"  *** WARNING: {name} single-pass accuracy ({acc_o:.4f}) is near chance "
              f"(chance={CHANCE:.4f}) -- the checkpoint may not have reloaded correctly. ***")
    tta_rows.append({"model": name, "n_test": len(y_true),
                      "acc_original_view": float(acc_o), "acc_flipped_view": float(acc_f),
                      "acc_tta_averaged": float(acc_t), "view_agreement_rate": float(agree),
                      "n_corrected_by_tta": corrected, "n_harmed_by_tta": harmed,
                      "n_unchanged_by_tta": unchanged, "tta_minus_original_delta": float(acc_t-acc_o)})
    print(f"  {name}: orig={acc_o:.4f} flip={acc_f:.4f} tta={acc_t:.4f} "
          f"agree={agree:.4f} corrected={corrected} harmed={harmed} unchanged={unchanged}")
    del model; torch.cuda.empty_cache()

tta_df = pd.DataFrame(tta_rows)
tta_df.to_csv(OUTDIR/"tta_breakdown_per_classifier.csv", index=False)
print("\nSaved: tta_breakdown_per_classifier.csv")
print(tta_df.to_string(index=False))

  MobileNetV3Large: orig=0.6874 flip=0.6925 tta=0.6906 agree=0.9404 corrected=18 harmed=13 unchanged=1530
  EfficientNetB0: orig=0.6515 flip=0.6586 tta=0.6560 agree=0.9289 corrected=21 harmed=14 unchanged=1526
  ResNet50: orig=0.6919 flip=0.6970 tta=0.6989 agree=0.9276 corrected=26 harmed=15 unchanged=1520
  EfficientNetV2S: orig=0.7335 flip=0.7290 tta=0.7322 agree=0.9161 corrected=20 harmed=22 unchanged=1519
  ConvNeXtV2Tiny: orig=0.6765 flip=0.6803 tta=0.6855 agree=0.9129 corrected=28 harmed=14 unchanged=1519
  ViTBase16: orig=0.0833 flip=0.0852 tta=0.0884 agree=0.5490 corrected=27 harmed=19 unchanged=1515
  *** WARNING: SwinTiny single-pass accuracy (0.0122) is near chance (chance=0.0087) -- the checkpoint may not have reloaded correctly. ***
  SwinTiny: orig=0.0122 flip=0.0109 tta=0.0122 agree=0.9110 corrected=0 harmed=0 unchanged=1561
  GatedFusionLight: orig=0.6778 flip=0.6855 tta=0.6874 agree=0.8757 corrected=42 harmed=27 unchanged=1492

Saved: tta_breakdown_per_classifier.csv
 

## Section 3 -- Leakage dedup + recompute on the full 311-image union

The paper's clean-vs-leaked comparison (Stage 6 Section 3) used only the 250-image URL-leak
list. R5 point 2 asks for the fuller picture: the union of the URL-leak list (250 images)
and the perceptual-hash near-duplicate audit (213 images, 152 shared with the URL-leak
list) is **311 images** -- exactly the number already reported in the paper's Limitations
section. This section reconstructs that union from its two source files and extends the
comparison from classification accuracy alone to segmentation fg-IoU, severity MAE/CCC, and
a confidence-flag re-validation restricted to the clean (non-flagged) subset only.

In [5]:
# ============ SECTION 3: LEAKAGE DEDUP + RECOMPUTE (311-IMAGE UNION) ============
# Reconstructed from the two independently-computed source files already in the project:
# crosssplit_duplicate_audit.csv (213 unique near-duplicate test images) and
# leaked_test_images.txt (250 URL-leak test images); 152 images are in both, so the union
# is exactly 311 -- matching the paper's reported number.
DUPAUDIT_NAMES = ['apple_black_rot_144.jpg', 'apple_scab_66.jpg', 'banana_black_leaf_streak_banana black sigatoka (42).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_cigar_end_rot_Bing_0002.jpg', 'banana_cigar_end_rot_Bing_0014.jpg', 'bean_mosaic_virus_30.jpg', 'bean_rust_google_0130.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0019.jpg', 'bell_pepper_bacterial_spot_Bing_0164.jpg', 'bell_pepper_blossom_end_rot_Bing_0008.jpg', 'bell_pepper_blossom_end_rot_Bing_0046.jpg', 'bell_pepper_blossom_end_rot_Bing_0112.jpg', 'bell_pepper_blossom_end_rot_Google_0044.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0019.jpg', 'bell_pepper_powdery_mildew_Bing_0001.jpg', 'blueberry_scorch_Bing_0011.jpg', 'broccoli_alternaria_leaf_spot_Bing_0019.jpg', 'broccoli_alternaria_leaf_spot_Bing_0021.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_alternaria_leaf_spot_Bing_0060.jpg', 'broccoli_alternaria_leaf_spot_Google_0005.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_google_0026.jpg', 'cabbage_alternaria_leaf_spot_google_0148.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0007.jpg', 'cabbage_black_rot_Bing_0218 - Copy.jpg', 'cabbage_black_rot_Google_0101.jpg', 'cabbage_black_rot_Google_0110.jpg', 'cabbage_downy_mildew_Bing_0039.jpg', 'carrot_alternaria_leaf_blight_Bing_0075.jpg', 'carrot_alternaria_leaf_blight_Bing_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'cauliflower_alternaria_leaf_spot_google_0028.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'celery_anthracnose_google_0035.jpg', 'citrus_canker_107.jpg', 'citrus_canker_163.jpg', 'citrus_canker_276.jpg', 'citrus_canker_285.jpg', 'citrus_canker_90.jpg', 'coffee_berry_blotch_Bing_0006.jpg', 'coffee_berry_blotch_Bing_0010.jpg', 'coffee_berry_blotch_Bing_0037.jpg', 'coffee_berry_blotch_Bing_0049.jpg', 'coffee_berry_blotch_Bing_0058.jpg', 'coffee_berry_blotch_Bing_0099.jpg', 'coffee_berry_blotch_Bing_0117.jpg', 'coffee_berry_blotch_Bing_0122.jpg', 'coffee_brown_eye_spot_Bing_0006.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_google_0048.jpg', 'coffee_leaf_rust_google_0125.jpg', 'coffee_leaf_rust_google_0130.jpg', 'cucumber_angular_leaf_spot_175.jpg', 'cucumber_bacterial_wilt_google_0076.jpg', 'cucumber_powdery_mildew_26.jpg', 'cucumber_powdery_mildew_google_0001.jpg', 'cucumber_powdery_mildew_google_0026.jpg', 'cucumber_powdery_mildew_google_0080.jpg', 'eggplant_cercospora_leaf_spot_96.jpg', 'eggplant_cercospora_leaf_spot_google_0030.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_rust_98.jpg', 'ginger_sheath_blight_13.jpg', 'ginger_sheath_blight_161.jpg', 'ginger_sheath_blight_30.jpg', 'ginger_sheath_blight_89.jpg', 'ginger_sheath_blight_google_0007.jpg', 'grape_black_rot_google_0320.jpg', 'grape_downy_mildew_241.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_64.jpg', 'grape_leaf_spot_157.jpg', 'grape_leaf_spot_168.jpg', 'grape_leaf_spot_9.jpg', 'peach_brown_rot_Bing_0040.jpg', 'peach_brown_rot_Bing_0141.jpg', 'peach_brown_rot_Bing_0269.jpg', 'peach_scab_Bing_0046.jpg', 'peach_scab_Bing_0054.jpg', 'peach_scab_Bing_0083.jpg', 'peach_scab_Bing_0223.jpg', 'plum_bacterial_spot_Google_0057.jpg', 'plum_brown_rot_Bing_0000.jpg', 'plum_brown_rot_Bing_0079.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0036.jpg', 'potato_early_blight_google_0004.jpg', 'potato_early_blight_google_0018.jpg', 'potato_late_blight_google_0048.jpg', 'raspberry_fire_blight_Bing_0041.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'rice_blast_151.jpg', 'rice_sheath_blight_52.jpg', 'rice_sheath_blight_google_0000.jpg', 'soybean_bacterial_blight_Bing_0049.jpg', 'soybean_bacterial_blight_Bing_0055.jpg', 'soybean_bacterial_blight_Bing_0066.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Bing_0025.jpg', 'soybean_brown_spot_Bing_0090.jpg', 'soybean_downy_mildew_Bing_0023.jpg', 'soybean_downy_mildew_Bing_0038.jpg', 'soybean_downy_mildew_Bing_0045.jpg', 'soybean_downy_mildew_Google_0061.jpg', 'soybean_frog_eye_leaf_spot_Baidu_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0004.jpg', 'soybean_frog_eye_leaf_spot_Bing_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0089.jpg', 'soybean_frog_eye_leaf_spot_Bing_0095.jpg', 'soybean_frog_eye_leaf_spot_Google_0048.jpg', 'soybean_frog_eye_leaf_spot_Google_0056.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_mosaic_Bing_0062.jpg', 'soybean_mosaic_Bing_0112.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0032.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'squash_powdery_mildew_17.jpg', 'squash_powdery_mildew_google_0006.jpg', 'squash_powdery_mildew_google_0079.jpg', 'squash_powdery_mildew_google_0102.jpg', 'squash_powdery_mildew_google_0191.jpg', 'squash_powdery_mildew_google_0327.jpg', 'tobacco_blue_mold_Bing_0107.jpg', 'tobacco_blue_mold_Bing_0127.jpg', 'tobacco_blue_mold_Google_0008.jpg', 'tobacco_brown_spot_Bing_0003.jpg', 'tobacco_brown_spot_Bing_0122.jpg', 'tobacco_brown_spot_Google_0029.jpg', 'tobacco_frogeye_leaf_spot_Bing_0000.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tomato_bacterial_leaf_spot_78.jpg', 'tomato_bacterial_leaf_spot_google_0163.jpg', 'tomato_early_blight_173.jpg', 'tomato_early_blight_google_0041.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0267.jpg', 'tomato_mosaic_virus_33.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_google_0205.jpg', 'tomato_yellow_leaf_curl_virus_4.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Bing_0004.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Google_0141.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_black_chaff (25).jpg', 'wheat_head_scab_Baidu_0065.jpg', 'wheat_head_scab_Baidu_0281.jpg', 'wheat_head_scab_Baidu_0310.jpg', 'wheat_head_scab_Baidu_0368.jpg', 'wheat_head_scab_Baidu_0404.jpg', 'wheat_head_scab_Baidu_0431.jpg', 'wheat_head_scab_Bing_0023.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0111.jpg', 'wheat_head_scab_Bing_0206.jpg', 'wheat_head_scab_Bing_0219.jpg', 'wheat_head_scab_Bing_0287.jpg', 'wheat_head_scab_Google_0083.jpg', 'wheat_head_scab_Google_0309.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0374.jpg', 'wheat_loose_smut_Bing_0008.jpg', 'wheat_loose_smut_Bing_0097.jpg', 'wheat_loose_smut_Google_0065.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0049.jpg', 'wheat_powdery_mildew_Bing_0023.jpg', 'wheat_powdery_mildew_Bing_0163.jpg', 'wheat_powdery_mildew_Bing_0170.jpg', 'wheat_powdery_mildew_Google_0101.jpg', 'wheat_septoria_blotch_Bing_0045.jpg', 'wheat_septoria_blotch_Google_0016.jpg', 'wheat_septoria_blotch_Google_0058.jpg', 'wheat_septoria_blotch_Google_0174.jpg', 'wheat_septoria_blotch_Google_0206.jpg', 'wheat_septoria_blotch_Google_0309.jpg', 'wheat_septoria_blotch_blotch (22).jpg', 'wheat_septoria_blotch_blotch (69).jpg', 'wheat_septoria_blotch_blotch (79).jpg', 'wheat_stem_rust_Baidu_0003.jpg', 'wheat_stem_rust_Bing_0002.jpg', 'wheat_stem_rust_Bing_0016.jpg', 'wheat_stem_rust_Bing_0040.jpg', 'wheat_stem_rust_Bing_0070.jpg', 'wheat_stem_rust_Bing_0206.jpg', 'wheat_stem_rust_Bing_0247.jpg', 'wheat_stem_rust_Google_0071.jpg', 'wheat_stripe_rust_Baidu_0106.jpg', 'wheat_stripe_rust_Baidu_0120.jpg', 'wheat_stripe_rust_Baidu_0317.jpg', 'wheat_stripe_rust_Baidu_0484.jpg', 'wheat_stripe_rust_Bing_0137.jpg', 'wheat_stripe_rust_Bing_0173.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'wheat_stripe_rust_Google_0223.jpg', 'zucchini_bacterial_wilt_Bing_0019.jpg', 'zucchini_powdery_mildew_Bing_0002.jpg', 'zucchini_powdery_mildew_Bing_0004.jpg', 'zucchini_powdery_mildew_Bing_0052.jpg', 'zucchini_powdery_mildew_Bing_0054.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_powdery_mildew_Bing_0289.jpg', 'zucchini_powdery_mildew_Bing_0337.jpg']
URLLEAK_NAMES = ['apple_black_rot_144.jpg', 'apple_mosaic_virus_43.jpg', 'apple_mosaic_virus_57.jpg', 'apple_rust_74.jpg', 'apple_rust_96.jpg', 'apple_scab_50.jpg', 'apple_scab_8.jpg', 'banana_black_leaf_streak_Baidu_0006.jpg', 'banana_black_leaf_streak_banana black sigatoka (42).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_cigar_end_rot_Bing_0002.jpg', 'banana_cigar_end_rot_Bing_0014.jpg', 'banana_panama_disease_6.jpg', 'bean_rust_178.jpg', 'bean_rust_189.jpg', 'bean_rust_google_0130.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0005.jpg', 'bell_pepper_blossom_end_rot_Bing_0008.jpg', 'bell_pepper_blossom_end_rot_Bing_0046.jpg', 'bell_pepper_blossom_end_rot_Bing_0112.jpg', 'bell_pepper_blossom_end_rot_Google_0044.jpg', 'bell_pepper_powdery_mildew_Bing_0001.jpg', 'blueberry_scorch_Bing_0011.jpg', 'broccoli_alternaria_leaf_spot_Bing_0019.jpg', 'broccoli_alternaria_leaf_spot_Bing_0021.jpg', 'broccoli_alternaria_leaf_spot_Bing_0045.jpg', 'broccoli_alternaria_leaf_spot_Bing_0060.jpg', 'broccoli_alternaria_leaf_spot_Google_0005.jpg', 'broccoli_downy_mildew_9.jpg', 'cabbage_alternaria_leaf_spot_48.jpg', 'cabbage_alternaria_leaf_spot_google_0026.jpg', 'cabbage_alternaria_leaf_spot_google_0148.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0007.jpg', 'cabbage_black_rot_Bing_0218 - Copy.jpg', 'cabbage_black_rot_Google_0101.jpg', 'cabbage_downy_mildew_Bing_0039.jpg', 'carrot_alternaria_leaf_blight_Bing_0075.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'cauliflower_alternaria_leaf_spot_google_0028.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'celery_anthracnose_google_0035.jpg', 'citrus_canker_107.jpg', 'citrus_canker_11.jpg', 'citrus_canker_130.jpg', 'citrus_canker_162.jpg', 'citrus_canker_163.jpg', 'citrus_canker_197.jpg', 'citrus_canker_251.jpg', 'citrus_canker_256.jpg', 'citrus_canker_276.jpg', 'citrus_canker_340.jpg', 'citrus_canker_388.jpg', 'citrus_canker_4.jpg', 'citrus_canker_50.jpg', 'citrus_canker_60.jpg', 'citrus_canker_76.jpg', 'coffee_berry_blotch_Bing_0006.jpg', 'coffee_berry_blotch_Bing_0010.jpg', 'coffee_berry_blotch_Bing_0037.jpg', 'coffee_berry_blotch_Bing_0049.jpg', 'coffee_berry_blotch_Bing_0099.jpg', 'coffee_berry_blotch_Bing_0117.jpg', 'coffee_berry_blotch_Bing_0140.jpg', 'coffee_brown_eye_spot_Bing_0006.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'corn_northern_leaf_blight_2.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_141.jpg', 'cucumber_angular_leaf_spot_261.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_bacterial_wilt_184.jpg', 'cucumber_bacterial_wilt_58.jpg', 'cucumber_powdery_mildew_10.jpg', 'cucumber_powdery_mildew_125.jpg', 'cucumber_powdery_mildew_178.jpg', 'cucumber_powdery_mildew_183.jpg', 'cucumber_powdery_mildew_198.jpg', 'cucumber_powdery_mildew_227.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0001.jpg', 'cucumber_powdery_mildew_google_0026.jpg', 'cucumber_powdery_mildew_google_0080.jpg', 'eggplant_cercospora_leaf_spot_96.jpg', 'eggplant_cercospora_leaf_spot_google_0030.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'garlic_leaf_blight_35.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_rust_56.jpg', 'garlic_rust_83.jpg', 'ginger_leaf_spot_161.jpg', 'ginger_sheath_blight_30.jpg', 'ginger_sheath_blight_89.jpg', 'ginger_sheath_blight_google_0007.jpg', 'grape_downy_mildew_115.jpg', 'grape_downy_mildew_202.jpg', 'grape_downy_mildew_223.jpg', 'grape_downy_mildew_241.jpg', 'grape_downy_mildew_29.jpg', 'grape_downy_mildew_49.jpg', 'grape_downy_mildew_82.jpg', 'lettuce_mosaic_virus_8.jpg', 'maple_tar_spot_google_0200.jpg', 'peach_brown_rot_Bing_0009.jpg', 'peach_brown_rot_Bing_0040.jpg', 'peach_brown_rot_Bing_0141.jpg', 'peach_brown_rot_Bing_0269.jpg', 'peach_leaf_curl_2.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_scab_Bing_0046.jpg', 'peach_scab_Bing_0083.jpg', 'peach_scab_Bing_0223.jpg', 'plum_bacterial_spot_Google_0057.jpg', 'plum_brown_rot_Bing_0000.jpg', 'plum_brown_rot_Bing_0079.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Google_0040.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_95.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0101.jpg', 'rice_sheath_blight_43.jpg', 'rice_sheath_blight_52.jpg', 'rice_sheath_blight_58.jpg', 'rice_sheath_blight_google_0000.jpg', 'soybean_bacterial_blight_Bing_0055.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_brown_spot_Bing_0025.jpg', 'soybean_brown_spot_Bing_0090.jpg', 'soybean_downy_mildew_Bing_0038.jpg', 'soybean_downy_mildew_Bing_0045.jpg', 'soybean_downy_mildew_Google_0061.jpg', 'soybean_frog_eye_leaf_spot_Baidu_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0004.jpg', 'soybean_frog_eye_leaf_spot_Bing_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0047.jpg', 'soybean_frog_eye_leaf_spot_Bing_0089.jpg', 'soybean_frog_eye_leaf_spot_Bing_0095.jpg', 'soybean_frog_eye_leaf_spot_Bing_0155.jpg', 'soybean_frog_eye_leaf_spot_Google_0048.jpg', 'soybean_frog_eye_leaf_spot_Google_0056.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_mosaic_Bing_0062.jpg', 'soybean_rust_Bing_0032.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'squash_powdery_mildew_114.jpg', 'squash_powdery_mildew_google_0006.jpg', 'squash_powdery_mildew_google_0079.jpg', 'squash_powdery_mildew_google_0102.jpg', 'squash_powdery_mildew_google_0191.jpg', 'tobacco_blue_mold_Bing_0107.jpg', 'tobacco_blue_mold_Bing_0127.jpg', 'tobacco_blue_mold_Google_0008.jpg', 'tobacco_brown_spot_Bing_0003.jpg', 'tobacco_brown_spot_Bing_0044.jpg', 'tobacco_brown_spot_Bing_0159.jpg', 'tobacco_brown_spot_Google_0029.jpg', 'tobacco_frogeye_leaf_spot_Bing_0000.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_mosaic_virus_161.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_bacterial_leaf_spot_31.jpg', 'tomato_bacterial_leaf_spot_78.jpg', 'tomato_bacterial_leaf_spot_google_0087.jpg', 'tomato_bacterial_leaf_spot_google_0163.jpg', 'tomato_early_blight_10.jpg', 'tomato_early_blight_136.jpg', 'tomato_early_blight_159.jpg', 'tomato_early_blight_4.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0041.jpg', 'tomato_early_blight_google_0267.jpg', 'tomato_late_blight_199.jpg', 'tomato_leaf_mold_182.jpg', 'tomato_leaf_mold_259.jpg', 'tomato_leaf_mold_59.jpg', 'tomato_mosaic_virus_55.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_63.jpg', 'tomato_septoria_leaf_spot_google_0205.jpg', 'tomato_yellow_leaf_curl_virus_4.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0202.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0217.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0275.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Google_0141.jpg', 'wheat_head_scab_Baidu_0065.jpg', 'wheat_head_scab_Baidu_0310.jpg', 'wheat_head_scab_Baidu_0368.jpg', 'wheat_head_scab_Baidu_0404.jpg', 'wheat_head_scab_Bing_0023.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0111.jpg', 'wheat_head_scab_Bing_0206.jpg', 'wheat_head_scab_Bing_0219.jpg', 'wheat_head_scab_Google_0083.jpg', 'wheat_head_scab_Google_0309.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0374.jpg', 'wheat_loose_smut_Baidu_0036.jpg', 'wheat_loose_smut_Bing_0008.jpg', 'wheat_loose_smut_Bing_0097.jpg', 'wheat_loose_smut_Google_0065.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0040.jpg', 'wheat_powdery_mildew_Baidu_0082.jpg', 'wheat_powdery_mildew_Baidu_0083.jpg', 'wheat_powdery_mildew_Bing_0023.jpg', 'wheat_powdery_mildew_Bing_0163.jpg', 'wheat_powdery_mildew_Google_0101.jpg', 'wheat_septoria_blotch_Baidu_0098.jpg', 'wheat_septoria_blotch_Bing_0045.jpg', 'wheat_septoria_blotch_Google_0016.jpg', 'wheat_septoria_blotch_Google_0058.jpg', 'wheat_septoria_blotch_Google_0174.jpg', 'wheat_septoria_blotch_Google_0206.jpg', 'wheat_septoria_blotch_Google_0309.jpg', 'wheat_septoria_blotch_blotch (22).jpg', 'wheat_septoria_blotch_blotch (69).jpg', 'wheat_septoria_blotch_blotch (79).jpg', 'wheat_stem_rust_Bing_0002.jpg', 'wheat_stem_rust_Bing_0016.jpg', 'wheat_stem_rust_Bing_0040.jpg', 'wheat_stem_rust_Bing_0070.jpg', 'wheat_stem_rust_Bing_0206.jpg', 'wheat_stem_rust_Bing_0247.jpg', 'wheat_stem_rust_Google_0071.jpg', 'wheat_stripe_rust_Baidu_0349.jpg', 'wheat_stripe_rust_Baidu_0401.jpg', 'wheat_stripe_rust_Baidu_0484.jpg', 'wheat_stripe_rust_Bing_0137.jpg', 'wheat_stripe_rust_Bing_0173.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'wheat_stripe_rust_Google_0223.jpg', 'zucchini_bacterial_wilt_Bing_0019.jpg', 'zucchini_powdery_mildew_Bing_0002.jpg', 'zucchini_powdery_mildew_Bing_0004.jpg', 'zucchini_powdery_mildew_Bing_0052.jpg', 'zucchini_powdery_mildew_Bing_0054.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0289.jpg', 'zucchini_powdery_mildew_Bing_0336.jpg', 'zucchini_powdery_mildew_Bing_0337.jpg', 'zucchini_powdery_mildew_Google_0045.jpg']
UNION_FLAGGED_NAMES = ['apple_black_rot_144.jpg', 'apple_mosaic_virus_43.jpg', 'apple_mosaic_virus_57.jpg', 'apple_rust_74.jpg', 'apple_rust_96.jpg', 'apple_scab_50.jpg', 'apple_scab_66.jpg', 'apple_scab_8.jpg', 'banana_black_leaf_streak_Baidu_0006.jpg', 'banana_black_leaf_streak_banana black sigatoka (42).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_cigar_end_rot_Bing_0002.jpg', 'banana_cigar_end_rot_Bing_0014.jpg', 'banana_panama_disease_6.jpg', 'bean_mosaic_virus_30.jpg', 'bean_rust_178.jpg', 'bean_rust_189.jpg', 'bean_rust_google_0130.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0005.jpg', 'bell_pepper_bacterial_spot_Bing_0019.jpg', 'bell_pepper_bacterial_spot_Bing_0164.jpg', 'bell_pepper_blossom_end_rot_Bing_0008.jpg', 'bell_pepper_blossom_end_rot_Bing_0046.jpg', 'bell_pepper_blossom_end_rot_Bing_0112.jpg', 'bell_pepper_blossom_end_rot_Google_0044.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0019.jpg', 'bell_pepper_powdery_mildew_Bing_0001.jpg', 'blueberry_scorch_Bing_0011.jpg', 'broccoli_alternaria_leaf_spot_Bing_0019.jpg', 'broccoli_alternaria_leaf_spot_Bing_0021.jpg', 'broccoli_alternaria_leaf_spot_Bing_0045.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_alternaria_leaf_spot_Bing_0060.jpg', 'broccoli_alternaria_leaf_spot_Google_0005.jpg', 'broccoli_downy_mildew_9.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_48.jpg', 'cabbage_alternaria_leaf_spot_google_0026.jpg', 'cabbage_alternaria_leaf_spot_google_0148.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0007.jpg', 'cabbage_black_rot_Bing_0218 - Copy.jpg', 'cabbage_black_rot_Google_0101.jpg', 'cabbage_black_rot_Google_0110.jpg', 'cabbage_downy_mildew_Bing_0039.jpg', 'carrot_alternaria_leaf_blight_Bing_0075.jpg', 'carrot_alternaria_leaf_blight_Bing_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'cauliflower_alternaria_leaf_spot_google_0028.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'celery_anthracnose_google_0035.jpg', 'citrus_canker_107.jpg', 'citrus_canker_11.jpg', 'citrus_canker_130.jpg', 'citrus_canker_162.jpg', 'citrus_canker_163.jpg', 'citrus_canker_197.jpg', 'citrus_canker_251.jpg', 'citrus_canker_256.jpg', 'citrus_canker_276.jpg', 'citrus_canker_285.jpg', 'citrus_canker_340.jpg', 'citrus_canker_388.jpg', 'citrus_canker_4.jpg', 'citrus_canker_50.jpg', 'citrus_canker_60.jpg', 'citrus_canker_76.jpg', 'citrus_canker_90.jpg', 'coffee_berry_blotch_Bing_0006.jpg', 'coffee_berry_blotch_Bing_0010.jpg', 'coffee_berry_blotch_Bing_0037.jpg', 'coffee_berry_blotch_Bing_0049.jpg', 'coffee_berry_blotch_Bing_0058.jpg', 'coffee_berry_blotch_Bing_0099.jpg', 'coffee_berry_blotch_Bing_0117.jpg', 'coffee_berry_blotch_Bing_0122.jpg', 'coffee_berry_blotch_Bing_0140.jpg', 'coffee_brown_eye_spot_Bing_0006.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_google_0048.jpg', 'coffee_leaf_rust_google_0125.jpg', 'coffee_leaf_rust_google_0130.jpg', 'corn_northern_leaf_blight_2.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_141.jpg', 'cucumber_angular_leaf_spot_175.jpg', 'cucumber_angular_leaf_spot_261.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_bacterial_wilt_184.jpg', 'cucumber_bacterial_wilt_58.jpg', 'cucumber_bacterial_wilt_google_0076.jpg', 'cucumber_powdery_mildew_10.jpg', 'cucumber_powdery_mildew_125.jpg', 'cucumber_powdery_mildew_178.jpg', 'cucumber_powdery_mildew_183.jpg', 'cucumber_powdery_mildew_198.jpg', 'cucumber_powdery_mildew_227.jpg', 'cucumber_powdery_mildew_26.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0001.jpg', 'cucumber_powdery_mildew_google_0026.jpg', 'cucumber_powdery_mildew_google_0080.jpg', 'eggplant_cercospora_leaf_spot_96.jpg', 'eggplant_cercospora_leaf_spot_google_0030.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_35.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_rust_56.jpg', 'garlic_rust_83.jpg', 'garlic_rust_98.jpg', 'ginger_leaf_spot_161.jpg', 'ginger_sheath_blight_13.jpg', 'ginger_sheath_blight_161.jpg', 'ginger_sheath_blight_30.jpg', 'ginger_sheath_blight_89.jpg', 'ginger_sheath_blight_google_0007.jpg', 'grape_black_rot_google_0320.jpg', 'grape_downy_mildew_115.jpg', 'grape_downy_mildew_202.jpg', 'grape_downy_mildew_223.jpg', 'grape_downy_mildew_241.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_29.jpg', 'grape_downy_mildew_49.jpg', 'grape_downy_mildew_64.jpg', 'grape_downy_mildew_82.jpg', 'grape_leaf_spot_157.jpg', 'grape_leaf_spot_168.jpg', 'grape_leaf_spot_9.jpg', 'lettuce_mosaic_virus_8.jpg', 'maple_tar_spot_google_0200.jpg', 'peach_brown_rot_Bing_0009.jpg', 'peach_brown_rot_Bing_0040.jpg', 'peach_brown_rot_Bing_0141.jpg', 'peach_brown_rot_Bing_0269.jpg', 'peach_leaf_curl_2.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_scab_Bing_0046.jpg', 'peach_scab_Bing_0054.jpg', 'peach_scab_Bing_0083.jpg', 'peach_scab_Bing_0223.jpg', 'plum_bacterial_spot_Google_0057.jpg', 'plum_brown_rot_Bing_0000.jpg', 'plum_brown_rot_Bing_0079.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0036.jpg', 'potato_early_blight_google_0004.jpg', 'potato_early_blight_google_0018.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_95.jpg', 'potato_late_blight_google_0048.jpg', 'raspberry_fire_blight_Bing_0041.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0101.jpg', 'rice_blast_151.jpg', 'rice_sheath_blight_43.jpg', 'rice_sheath_blight_52.jpg', 'rice_sheath_blight_58.jpg', 'rice_sheath_blight_google_0000.jpg', 'soybean_bacterial_blight_Bing_0049.jpg', 'soybean_bacterial_blight_Bing_0055.jpg', 'soybean_bacterial_blight_Bing_0066.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Bing_0025.jpg', 'soybean_brown_spot_Bing_0090.jpg', 'soybean_downy_mildew_Bing_0023.jpg', 'soybean_downy_mildew_Bing_0038.jpg', 'soybean_downy_mildew_Bing_0045.jpg', 'soybean_downy_mildew_Google_0061.jpg', 'soybean_frog_eye_leaf_spot_Baidu_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0004.jpg', 'soybean_frog_eye_leaf_spot_Bing_0012.jpg', 'soybean_frog_eye_leaf_spot_Bing_0047.jpg', 'soybean_frog_eye_leaf_spot_Bing_0089.jpg', 'soybean_frog_eye_leaf_spot_Bing_0095.jpg', 'soybean_frog_eye_leaf_spot_Bing_0155.jpg', 'soybean_frog_eye_leaf_spot_Google_0048.jpg', 'soybean_frog_eye_leaf_spot_Google_0056.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_mosaic_Bing_0062.jpg', 'soybean_mosaic_Bing_0112.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0032.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'squash_powdery_mildew_114.jpg', 'squash_powdery_mildew_17.jpg', 'squash_powdery_mildew_google_0006.jpg', 'squash_powdery_mildew_google_0079.jpg', 'squash_powdery_mildew_google_0102.jpg', 'squash_powdery_mildew_google_0191.jpg', 'squash_powdery_mildew_google_0327.jpg', 'tobacco_blue_mold_Bing_0107.jpg', 'tobacco_blue_mold_Bing_0127.jpg', 'tobacco_blue_mold_Google_0008.jpg', 'tobacco_brown_spot_Bing_0003.jpg', 'tobacco_brown_spot_Bing_0044.jpg', 'tobacco_brown_spot_Bing_0122.jpg', 'tobacco_brown_spot_Bing_0159.jpg', 'tobacco_brown_spot_Google_0029.jpg', 'tobacco_frogeye_leaf_spot_Bing_0000.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_mosaic_virus_161.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_bacterial_leaf_spot_31.jpg', 'tomato_bacterial_leaf_spot_78.jpg', 'tomato_bacterial_leaf_spot_google_0087.jpg', 'tomato_bacterial_leaf_spot_google_0163.jpg', 'tomato_early_blight_10.jpg', 'tomato_early_blight_136.jpg', 'tomato_early_blight_159.jpg', 'tomato_early_blight_173.jpg', 'tomato_early_blight_4.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0041.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0267.jpg', 'tomato_late_blight_199.jpg', 'tomato_leaf_mold_182.jpg', 'tomato_leaf_mold_259.jpg', 'tomato_leaf_mold_59.jpg', 'tomato_mosaic_virus_33.jpg', 'tomato_mosaic_virus_55.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_63.jpg', 'tomato_septoria_leaf_spot_google_0205.jpg', 'tomato_yellow_leaf_curl_virus_4.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0202.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0217.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0275.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Bing_0004.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Google_0141.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_black_chaff (25).jpg', 'wheat_head_scab_Baidu_0065.jpg', 'wheat_head_scab_Baidu_0281.jpg', 'wheat_head_scab_Baidu_0310.jpg', 'wheat_head_scab_Baidu_0368.jpg', 'wheat_head_scab_Baidu_0404.jpg', 'wheat_head_scab_Baidu_0431.jpg', 'wheat_head_scab_Bing_0023.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0111.jpg', 'wheat_head_scab_Bing_0206.jpg', 'wheat_head_scab_Bing_0219.jpg', 'wheat_head_scab_Bing_0287.jpg', 'wheat_head_scab_Google_0083.jpg', 'wheat_head_scab_Google_0309.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0374.jpg', 'wheat_loose_smut_Baidu_0036.jpg', 'wheat_loose_smut_Bing_0008.jpg', 'wheat_loose_smut_Bing_0097.jpg', 'wheat_loose_smut_Google_0065.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0040.jpg', 'wheat_powdery_mildew_Baidu_0049.jpg', 'wheat_powdery_mildew_Baidu_0082.jpg', 'wheat_powdery_mildew_Baidu_0083.jpg', 'wheat_powdery_mildew_Bing_0023.jpg', 'wheat_powdery_mildew_Bing_0163.jpg', 'wheat_powdery_mildew_Bing_0170.jpg', 'wheat_powdery_mildew_Google_0101.jpg', 'wheat_septoria_blotch_Baidu_0098.jpg', 'wheat_septoria_blotch_Bing_0045.jpg', 'wheat_septoria_blotch_Google_0016.jpg', 'wheat_septoria_blotch_Google_0058.jpg', 'wheat_septoria_blotch_Google_0174.jpg', 'wheat_septoria_blotch_Google_0206.jpg', 'wheat_septoria_blotch_Google_0309.jpg', 'wheat_septoria_blotch_blotch (22).jpg', 'wheat_septoria_blotch_blotch (69).jpg', 'wheat_septoria_blotch_blotch (79).jpg', 'wheat_stem_rust_Baidu_0003.jpg', 'wheat_stem_rust_Bing_0002.jpg', 'wheat_stem_rust_Bing_0016.jpg', 'wheat_stem_rust_Bing_0040.jpg', 'wheat_stem_rust_Bing_0070.jpg', 'wheat_stem_rust_Bing_0206.jpg', 'wheat_stem_rust_Bing_0247.jpg', 'wheat_stem_rust_Google_0071.jpg', 'wheat_stripe_rust_Baidu_0106.jpg', 'wheat_stripe_rust_Baidu_0120.jpg', 'wheat_stripe_rust_Baidu_0317.jpg', 'wheat_stripe_rust_Baidu_0349.jpg', 'wheat_stripe_rust_Baidu_0401.jpg', 'wheat_stripe_rust_Baidu_0484.jpg', 'wheat_stripe_rust_Bing_0137.jpg', 'wheat_stripe_rust_Bing_0173.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'wheat_stripe_rust_Google_0223.jpg', 'zucchini_bacterial_wilt_Bing_0019.jpg', 'zucchini_powdery_mildew_Bing_0002.jpg', 'zucchini_powdery_mildew_Bing_0004.jpg', 'zucchini_powdery_mildew_Bing_0052.jpg', 'zucchini_powdery_mildew_Bing_0054.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_powdery_mildew_Bing_0289.jpg', 'zucchini_powdery_mildew_Bing_0336.jpg', 'zucchini_powdery_mildew_Bing_0337.jpg', 'zucchini_powdery_mildew_Google_0045.jpg']
assert len(DUPAUDIT_NAMES) == 213 and len(URLLEAK_NAMES) == 250 and len(UNION_FLAGGED_NAMES) == 311
print(f"Flagged/leaked union: {len(UNION_FLAGGED_NAMES)} images "
      f"({len(DUPAUDIT_NAMES)} near-duplicate + {len(URLLEAK_NAMES)} URL-leak, "
      f"{len(set(DUPAUDIT_NAMES) & set(URLLEAK_NAMES))} overlap)")
FLAGGED_SET = set(UNION_FLAGGED_NAMES)
test_df = test_df.copy()
test_df["is_flagged"] = test_df["Name"].isin(FLAGGED_SET)
print(f"Test set: {(~test_df.is_flagged).sum()} clean / {test_df.is_flagged.sum()} flagged "
      f"(of {len(test_df)} total)")

def ccc_fn(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mt, mp = y_true.mean(), y_pred.mean(); vt, vp = y_true.var(), y_pred.var()
    covar = np.mean((y_true-mt)*(y_pred-mp))
    return (2*covar)/(vt+vp+(mt-mp)**2)

clf3 = build_classifier(NUM_CLASSES); clf3 = load_state(clf3, CLS_CKPT).to(device).eval()
seg3 = SegNet("DeepLabV3Plus_efficientnet-b3", 2); seg3 = load_state(seg3, SEG_CKPT_BEST).to(device).eval()
cam_extractor3 = GradCAM(model=clf3, target_layers=[clf3.stages[-1]])

leak_rows = []
for _, r in test_df.iterrows():
    img = Image.open(r["filepath"]).convert("RGB")
    x_cls = cls_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(clf3(x_cls), 1)[0]
    pred_idx = int(probs.argmax()); pred_class = CLASS_NAMES[pred_idx]
    correct = bool(pred_class == r["Disease"])

    x_seg = seg_eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_mask = seg3(x_seg).argmax(1)[0].cpu().numpy().astype(np.uint8)
    gt_raw = np.array(Image.open(r["maskpath"]))
    gt_bin_seg = np.array(Image.fromarray((gt_raw > 0).astype(np.uint8)*255).resize(
        (IMG_SIZE_SEG, IMG_SIZE_SEG), Image.NEAREST)) > 0
    inter = np.logical_and(pred_mask > 0, gt_bin_seg).sum(); union = np.logical_or(pred_mask > 0, gt_bin_seg).sum()
    fg_iou = float(inter/union) if union > 0 else np.nan

    severity_pred = float((pred_mask > 0).mean())
    severity_gt = float(r["Mask ratio"]) if "Mask ratio" in r and pd.notna(r["Mask ratio"]) else np.nan
    severity_abs_err = abs(severity_pred - severity_gt) if pd.notna(severity_gt) else np.nan

    cam = cam_extractor3(input_tensor=x_cls, targets=None)[0]
    cam_rs = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE_SEG, IMG_SIZE_SEG))) / 255.0
    tau = max(severity_pred, 1e-3)
    thresh = np.quantile(cam_rs, 1 - tau)
    cam_bin = (cam_rs >= thresh)
    inter2 = np.logical_and(cam_bin, pred_mask > 0).sum(); union2 = np.logical_or(cam_bin, pred_mask > 0).sum()
    cam_pred_iou = float(inter2/union2) if union2 > 0 else 0.0
    peak_yx = np.unravel_index(np.argmax(cam_rs), cam_rs.shape)
    pointing_hit = bool(pred_mask[peak_yx] > 0)
    confidence_flag = "HIGH" if pointing_hit else "LOW"

    leak_rows.append({"image": r["Name"], "is_flagged": bool(r["Name"] in FLAGGED_SET),
                       "correct": correct, "fg_iou": fg_iou,
                       "severity_pred": severity_pred, "severity_gt": severity_gt,
                       "severity_abs_error": severity_abs_err,
                       "cam_pred_iou": cam_pred_iou, "pointing_hit": pointing_hit,
                       "confidence_flag": confidence_flag})

leak_df = pd.DataFrame(leak_rows)
leak_df.to_csv(OUTDIR/"leakage_recompute_311union_per_image.csv", index=False)
del clf3, seg3, cam_extractor3; torch.cuda.empty_cache()

print("=== (a) Classification accuracy: clean vs. 311-image flagged union ===")
grp = leak_df.groupby("is_flagged")["correct"].agg(["mean","count"])
print(grp)
ct = pd.crosstab(leak_df.is_flagged, leak_df.correct)
odds, p_cls = fisher_exact(ct.values)
print(f"Fisher's exact (classification, flagged vs. clean): odds={odds:.3f} p={p_cls:.4f}")

print("\n=== (b) Segmentation fg-IoU: clean vs. flagged ===")
grp_iou = leak_df.groupby("is_flagged")["fg_iou"].agg(["mean","count"])
print(grp_iou)
u_stat, p_iou = mannwhitneyu(leak_df.loc[~leak_df.is_flagged, "fg_iou"].dropna(),
                              leak_df.loc[leak_df.is_flagged, "fg_iou"].dropna(), alternative="two-sided")
print(f"Mann-Whitney U (fg-IoU, clean vs. flagged): U={u_stat:.1f} p={p_iou:.4f}")

print("\n=== (c) Severity MAE / CCC: clean vs. flagged ===")
for flag_val, label in [(False, "clean"), (True, "flagged (311-union)")]:
    sub = leak_df[leak_df.is_flagged == flag_val].dropna(subset=["severity_gt"])
    mae = sub["severity_abs_error"].mean(); ccc = ccc_fn(sub["severity_gt"], sub["severity_pred"])
    print(f"  severity ({label}, n={len(sub)}): MAE={mae:.4f}  CCC={ccc:.4f}")

print("\n=== (d) Confidence-flag re-validation, CLEAN subset only ===")
clean_log = leak_df[~leak_df.is_flagged].copy()
summary_clean = clean_log.groupby("confidence_flag").agg(
    n=("image","count"), accuracy=("correct","mean"),
    mean_severity_abs_error=("severity_abs_error","mean"),
    mean_cam_pred_iou=("cam_pred_iou","mean")).round(4)
print(summary_clean)
ct2 = pd.crosstab(clean_log.confidence_flag, clean_log.correct)
p_flag = np.nan
if ct2.shape == (2,2):
    odds2, p_flag = fisher_exact(ct2.values)
    print(f"Fisher's exact (flag vs. correctness, clean-only): odds={odds2:.3f} p={p_flag:.4f}")
high_err = clean_log.loc[clean_log.confidence_flag=="HIGH", "severity_abs_error"].dropna()
low_err  = clean_log.loc[clean_log.confidence_flag=="LOW",  "severity_abs_error"].dropna()
if len(high_err) > 0 and len(low_err) > 0:
    u3, p3 = mannwhitneyu(high_err, low_err, alternative="two-sided")
    print(f"Mann-Whitney U (severity error, HIGH vs. LOW, clean-only): U={u3:.1f} p={p3:.4f}")

leak_summary = {
    "n_total": int(len(leak_df)), "n_flagged": int(leak_df.is_flagged.sum()),
    "n_clean": int((~leak_df.is_flagged).sum()),
    "classification_accuracy_flagged": float(grp.loc[True,"mean"]) if True in grp.index else None,
    "classification_accuracy_clean": float(grp.loc[False,"mean"]) if False in grp.index else None,
    "classification_fisher_p": float(p_cls),
    "fg_iou_mean_flagged": float(grp_iou.loc[True,"mean"]) if True in grp_iou.index else None,
    "fg_iou_mean_clean": float(grp_iou.loc[False,"mean"]) if False in grp_iou.index else None,
    "fg_iou_mannwhitney_p": float(p_iou),
    "confidence_flag_fisher_p_clean_only": float(p_flag) if pd.notna(p_flag) else None,
}
json.dump(leak_summary, open(OUTDIR/"leakage_recompute_SUMMARY.json", "w"), indent=2)
print(f"\nSaved: leakage_recompute_311union_per_image.csv and leakage_recompute_SUMMARY.json")

Flagged/leaked union: 311 images (213 near-duplicate + 250 URL-leak, 152 overlap)
Test set: 1250 clean / 311 flagged (of 1561 total)
=== (a) Classification accuracy: clean vs. 311-image flagged union ===
               mean  count
is_flagged                
False       0.67840   1250
True        0.66881    311
Fisher's exact (classification, flagged vs. clean): odds=0.957 p=0.7865

=== (b) Segmentation fg-IoU: clean vs. flagged ===
                mean  count
is_flagged                 
False       0.557852   1250
True        0.602781    311
Mann-Whitney U (fg-IoU, clean vs. flagged): U=176830.5 p=0.0137

=== (c) Severity MAE / CCC: clean vs. flagged ===
  severity (clean, n=1250): MAE=0.0703  CCC=0.7967
  severity (flagged (311-union), n=311): MAE=0.0625  CCC=0.8742

=== (d) Confidence-flag re-validation, CLEAN subset only ===
                   n  accuracy  mean_severity_abs_error  mean_cam_pred_iou
confidence_flag                                                           
HIGH      

## Section 4 -- Minor lookups answered by existing code (no new computation)

Two of the round-2 minor comments are fully answered by code that already exists and does
not need re-running -- they are documented here, with exact pointers, rather than padded
with a redundant cell.

**R5 point 1(d) -- how is the CAM-mask IoU threshold chosen for the deployed confidence
flag?** It is *not* tuned on any dataset. In the Stage 5 decision-support pipeline
(`diagnose_image()`), the Grad-CAM is binarized at
`thresh = np.quantile(cam_rs, 1 - tau)` where `tau = max(severity, 1e-3)` and `severity` is
the model's **own predicted** lesion-area fraction for that image, computed at inference
time. The identical line is reused in Section 3 above (`tau = max(severity_pred, 1e-3)`).
There is no fixed/global threshold and nothing is fit against ground truth or against the
test set as a whole.

**R5 minor comment (c) -- deletion-AUC pixel-aggregation rule.** The faithfulness
`deletion_auc` function (Stage 4, reused in Sections 1 above) tracks model confidence as
`torch.softmax(logits, 1)[0, 1:].sum(0).mean()` for the segmenters -- i.e. the **mean lesion
probability over every pixel in the image**, not just the pixels inside the predicted
lesion region. `deletion_auc_generic`'s `prob_fn` closures in Section 1 implement the same
rule (and the classifier variant tracks the single predicted-class probability, the natural
per-image analogue).

## Wrap-up -- zip all outputs for download

In [6]:
import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookA_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"Archive: {zip_path.name}  ({zip_path.stat().st_size/1024/1024:.2f} MB)")
print("Files included:")
for p in sorted(OUTDIR.glob("*")):
    print(" ", p.name)
print("\nDownload from the Kaggle Output tab and bring all CSV/JSON files back for the response letter.")

Archive: plantseg_r2_notebookA_results.zip  (0.16 MB)
Files included:
  chance_baseline_SUMMARY.csv
  chance_baseline_per_image.csv
  leakage_recompute_311union_per_image.csv
  leakage_recompute_SUMMARY.json
  tta_breakdown_per_classifier.csv

Download from the Kaggle Output tab and bring all CSV/JSON files back for the response letter.
